# Workflows vs Agents: When NOT to Use an Agent

**Phase 04** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-04/02-workflows-vs-agents.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 04-02: Workflows vs Agents: When NOT to Use an Agent
Phase 04: Agents - Patterns That Survive Production

Implements the same task two ways:
  - As a fixed workflow (1 LLM call, deterministic, cheap)
  - As an agent (multiple LLM calls, flexible, expensive)

Shows the cost/latency difference and a decision function for classifying tasks.
"""

import anthropic
import json
import time
from dataclasses import dataclass

client = anthropic.Anthropic()

### Implementation A: Fixed Workflow (correct for structured tasks)

In [ ]:
def summarize_review_workflow(review: str) -> dict:
    """
    Fixed workflow: one structured prompt, one API call.
    The LLM executes all steps in a single call.
    Python controls the flow. Output is structured JSON.
    """
    prompt = f"""Analyze this product review and return a JSON object with exactly these fields:
- sentiment: "positive" | "negative" | "neutral" | "mixed"
- themes: list of 2-4 key themes mentioned
- summary: one sentence capturing the main point

Review:
{review}

Return only valid JSON, no explanation, no markdown code fences."""

    start = time.time()
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start

    result = json.loads(response.content[0].text)
    result["_meta"] = {
        "approach": "workflow",
        "api_calls": 1,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "latency_ms": round(elapsed * 1000)
    }
    return result

### Implementation B: Agent (incorrect for this task, shown for comparison)

In [ ]:
ANALYSIS_TOOLS = [
    {
        "name": "extract_sentiment",
        "description": "Determine the overall sentiment of a text passage.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to analyze"}
            },
            "required": ["text"]
        }
    },
    {
        "name": "extract_themes",
        "description": "Extract 2-4 key themes from a text passage, returned as a comma-separated list.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to analyze"}
            },
            "required": ["text"]
        }
    },
    {
        "name": "write_summary",
        "description": "Write a one-sentence summary of a text passage.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to summarize"}
            },
            "required": ["text"]
        }
    }
]


def _tool_extract_sentiment(text: str) -> str:
    resp = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=32,
        messages=[{"role": "user", "content": f"Sentiment of this text (one word: positive/negative/neutral/mixed):\n{text}"}]
    )
    return resp.content[0].text.strip()


def _tool_extract_themes(text: str) -> str:
    resp = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=128,
        messages=[{"role": "user", "content": f"List 2-4 key themes, comma-separated, no explanation:\n{text}"}]
    )
    return resp.content[0].text.strip()


def _tool_write_summary(text: str) -> str:
    resp = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=64,
        messages=[{"role": "user", "content": f"Write one sentence summarizing the main point:\n{text}"}]
    )
    return resp.content[0].text.strip()


AGENT_TOOL_REGISTRY = {
    "extract_sentiment": lambda args: _tool_extract_sentiment(args["text"]),
    "extract_themes": lambda args: _tool_extract_themes(args["text"]),
    "write_summary": lambda args: _tool_write_summary(args["text"]),
}


def summarize_review_agent(review: str) -> dict:
    """
    Agent approach: the LLM decides which tools to call and in what order.
    Wrong for this task. Steps are fixed. This adds unnecessary loops and cost.
    """
    messages = [
        {"role": "user", "content": f"Analyze this product review. Extract the sentiment, key themes, and write a one-sentence summary:\n\n{review}"}
    ]
    system = "You are a review analyst. Use the available tools to extract sentiment, themes, and a summary from the review."

    start = time.time()
    api_calls = 1  # count the first orchestration call
    total_input = 0
    total_output = 0
    turns = 0

    for _ in range(10):
        response = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=512,
            system=system,
            tools=ANALYSIS_TOOLS,
            messages=messages
        )
        total_input += response.usage.input_tokens
        total_output += response.usage.output_tokens
        turns += 1

        if response.stop_reason == "end_turn":
            elapsed = time.time() - start
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
                    break
            return {
                "result": final_text,
                "_meta": {
                    "approach": "agent",
                    "api_calls": api_calls,
                    "turns": turns,
                    "input_tokens": total_input,
                    "output_tokens": total_output,
                    "latency_ms": round((time.time() - start) * 1000)
                }
            }

        if response.stop_reason == "tool_use":
            tool_blocks = [b for b in response.content if b.type == "tool_use"]
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for b in tool_blocks:
                api_calls += 1  # each tool call makes an LLM call in this implementation
                output = AGENT_TOOL_REGISTRY[b.name](b.input)
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": output})
            messages.append({"role": "user", "content": results})

    elapsed = time.time() - start
    return {
        "result": "max iterations reached",
        "_meta": {"approach": "agent", "api_calls": api_calls, "turns": turns,
                  "input_tokens": total_input, "output_tokens": total_output,
                  "latency_ms": round(elapsed * 1000)}
    }

### Decision framework: classify tasks before building

In [ ]:
@dataclass
class TaskProfile:
    """Profile a task before choosing its architecture."""
    fixed_steps: bool                  # Can you enumerate every step before it runs?
    predictable_branches: bool         # Are all branch conditions known before execution?
    needs_realtime_decisions: bool     # Must the LLM see intermediate results to decide next steps?
    state_spans_sessions: bool         # Does task state persist across multiple conversations?
    multiple_specialized_goals: bool   # Does it require coordination across distinct sub-agents?


def should_use_agent(profile: TaskProfile) -> str:
    """
    Returns 'workflow' | 'agent' | 'multi-agent'.
    Encode this decision in design review, not after shipping.
    """
    if profile.state_spans_sessions or profile.multiple_specialized_goals:
        return "multi-agent"

    if profile.fixed_steps and profile.predictable_branches:
        return "workflow"

    if profile.needs_realtime_decisions:
        return "agent"

    # Default to workflow when in doubt
    return "workflow"

### Demo

In [ ]:
SAMPLE_REVIEW = """
I've been using this laptop for three months now and have mixed feelings.
The battery life is excellent - I get a full 10 hours on a single charge, which is great
for travel. The display is also stunning with vibrant colors. However, the keyboard feels
mushy and lacks proper feedback, making long typing sessions uncomfortable. The fan
noise is another issue - it kicks in frequently even for light tasks. Customer support
was responsive when I had a setup issue, which I appreciated. Overall, good hardware
choices but some quality control issues in the input department.
"""

### Demo

In [ ]:
print("=" * 60)
print("APPROACH A: Fixed Workflow")
print("=" * 60)
workflow_result = summarize_review_workflow(SAMPLE_REVIEW)
meta = workflow_result.pop("_meta")
print(json.dumps(workflow_result, indent=2))
print(f"\nMetrics: {meta['api_calls']} API call(s) | "
      f"{meta['input_tokens']} in / {meta['output_tokens']} out tokens | "
      f"{meta['latency_ms']}ms")

print("\n" + "=" * 60)
print("APPROACH B: Agent Loop")
print("=" * 60)
agent_result = summarize_review_agent(SAMPLE_REVIEW)
meta = agent_result.pop("_meta")
print(f"Result: {agent_result.get('result', '')[:200]}")
print(f"\nMetrics: {meta['api_calls']} API call(s) | "
      f"{meta['turns']} turns | "
      f"{meta['input_tokens']} in / {meta['output_tokens']} out tokens | "
      f"{meta['latency_ms']}ms")

print("\n" + "=" * 60)
print("DECISION FRAMEWORK: Classify tasks before building")
print("=" * 60)

examples = [
    ("Document summarization", TaskProfile(True, True, False, False, False)),
    ("Customer support triage", TaskProfile(False, False, True, False, False)),
    ("Research assistant", TaskProfile(False, False, True, True, True)),
    ("Data validation pipeline", TaskProfile(True, True, False, False, False)),
    ("Debugging assistant", TaskProfile(False, False, True, False, False)),
]

for name, profile in examples:
    decision = should_use_agent(profile)
    print(f"  {name:<35} -> {decision}")

---

## Self-Check

Answer these without running code first:

**1. What is the correct diagnosis and fix?**

- A. The agent needs a better system prompt so it calls fewer tools.
- B. The task is structurally a workflow. The steps are fixed and enumerable. Rewrite as a deterministic pipeline with one structured prompt per step or a single multi-step prompt.
- C. The agent should use a more powerful model so it can combine steps more efficiently.
- D. Add a max_iterations guard of 3 to limit the number of tool calls.

**2. Which architecture is most appropriate?**

- A. Multi-agent: one agent for billing, one for technical, a third to coordinate.
- B. Workflow: classify the message with one LLM call, then branch to the appropriate template with a second LLM call.
- C. Agent: let the model decide dynamically which response to give, with tools for fetching policies and guides.
- D. Workflow: use a regex to classify the message, then an LLM call for the response.

**3. What is the structural property that makes this an agent rather than a workflow?**

- A. It uses web search, which requires a tool.
- B. The number of steps is unknown until execution begins and each step's next action depends on what was found in the current step.
- C. It produces a synthesis, which is too complex for a single prompt.
- D. It requires reading multiple sources, which cannot be done in one prompt.

**4. What is the correct engineering response?**

- A. Agree - building the agent version now is a reasonable hedge against future complexity.
- B. Build the workflow now. If the task evolves to require real-time decisions, refactoring from workflow to agent is a small change. The cost of running an unnecessary agent loop in production for months is not recoverable.
- C. Build both versions and run them in parallel, routing to the agent only when the workflow fails.
- D. Ask the PM to specify exactly what the more complex future version will look like before deciding.

**5. What is the right next step?**

- A. Switch the entire task to an agent so it can handle the edge cases dynamically.
- B. Handle the 15% edge case explicitly in the workflow with an additional branch or a fallback LLM call for that specific condition.
- C. Drop the 15% of inputs that cause errors and only process the 85% that work.
- D. Use a more powerful model that is less likely to encounter the edge case.

**6. What is the specific debuggability advantage that a workflow implementation would have had in this incident?**

- A. A workflow would have prevented the wrong answer entirely.
- B. A workflow would have been faster to fix because it uses fewer tokens.
- C. A workflow's steps are explicit Python functions with inspectable inputs and outputs. The bug would be localized to a specific step, not distributed across 12 tool calls where the model's intermediate reasoning is opaque.
- D. A workflow would have generated better error messages.

**7. Which architecture and why?**

- A. Agent: the model should decide how to present the data based on what it finds.
- B. Multi-agent: one agent for data, one for formatting, one for email.
- C. Workflow: all steps are fixed and enumerable. Use Python for data and formatting, one LLM call for any natural-language sections, a Python library for email.
- D. Agent: the report needs to be personalized, which requires dynamic decisions.

**8. What is the monthly cost difference, and what additional argument strengthens the case beyond cost?**

- A. $750/month difference. The additional argument is that the workflow is faster.
- B. $750/month difference. The additional argument is that the workflow is fully testable with deterministic assertions, enabling a proper regression suite, while the agent requires probabilistic evals.
- C. $1,500/month difference. The additional argument is that agents are harder to scale.
- D. $900/month difference. The additional argument is that the workflow uses fewer HTTP connections.

_Answers are in `checks.json` in the lesson directory._